# Kruskal–Wallis Test + Dunn's Post-Hoc Test

This notebook performs a **Kruskal–Wallis test** for three independent trading strategies and, when the overall test is significant, performs **Dunn's post-hoc test with Holm correction** to identify which pairs of strategies differ.

### Decision logic

- If `p >= alpha`: **Fail to reject H₀**.
- If `p < alpha`: **Reject H₀** and perform Dunn's post-hoc test.
- For Dunn's test, use the **adjusted p-value** to decide which pairs differ.

In [ ]:
# Install required packages if needed
# Uncomment the following line in a fresh environment:
# %pip install scipy scikit-posthocs pandas numpy

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import scikit_posthocs as sp

alpha = 0.05

## 1. Create sample data

The following are illustrative daily returns for three independent trading strategies.

The Kruskal–Wallis test is appropriate here when we want a rank-based comparison and do not want to rely on the normality/equal-variance assumptions of ordinary one-way ANOVA.

In [ ]:
strategy_a = np.array([
    0.10, 0.12, 0.08, 0.11, 0.09, 0.13, 0.07, 0.10, 0.11, 0.09,
    0.12, 0.08
])

strategy_b = np.array([
    0.11, 0.13, 0.10, 0.12, 0.09, 0.14, 0.08, 0.11, 0.12, 0.10,
    0.13, 0.09
])

strategy_c = np.array([
    0.20, 0.25, 0.22, 0.28, 0.24, 0.27, 0.23, 0.26, 0.21, 0.29,
    0.25, 0.24
])

print("Strategy A:", strategy_a)
print("Strategy B:", strategy_b)
print("Strategy C:", strategy_c)

## 2. Kruskal–Wallis test

### Hypotheses

**H₀:** The groups have the same distribution.

**H₁:** At least one group differs.

The test is performed with `scipy.stats.kruskal()`.

In [ ]:
h_stat, p_value = stats.kruskal(
    strategy_a,
    strategy_b,
    strategy_c
)

print(f"Kruskal-Wallis H statistic: {h_stat:.4f}")
print(f"p-value: {p_value:.6f}")
print(f"alpha: {alpha:.2f}")

In [ ]:
# Decision

if p_value < alpha:
    print("Decision: Reject H0")
    print("Conclusion: There is statistically significant evidence that at least one group differs.")
else:
    print("Decision: Fail to reject H0")
    print("Conclusion: There is insufficient statistical evidence to conclude that the groups differ.")

## 3. Dunn's post-hoc test

The Kruskal–Wallis test only tells us that **at least one group differs**. It does not identify the specific pairs.

When the Kruskal–Wallis test is significant, use Dunn's test for pairwise comparisons.

Here we use **Holm correction** to account for multiple comparisons.

In [ ]:
# Put all observations into long-format data
df = pd.DataFrame({
    "return": np.concatenate([strategy_a, strategy_b, strategy_c]),
    "strategy": (
        ["Strategy A"] * len(strategy_a)
        + ["Strategy B"] * len(strategy_b)
        + ["Strategy C"] * len(strategy_c)
    )
})

df.head()

In [ ]:
if p_value < alpha:
    dunn_results = sp.posthoc_dunn(
        df,
        val_col="return",
        group_col="strategy",
        p_adjust="holm"
    )

    print("Dunn's test with Holm correction:")
    display(dunn_results)
else:
    print("Kruskal-Wallis test is not significant.")
    print("Dunn's post-hoc test is not performed.")

## 4. Make a decision for every pair

For each pair:

- Adjusted p-value `< 0.05` → **Reject pairwise H₀**
- Adjusted p-value `>= 0.05` → **Fail to reject pairwise H₀**

In [ ]:
if p_value < alpha:
    groups = ["Strategy A", "Strategy B", "Strategy C"]

    pairwise_decisions = []

    for i in range(len(groups)):
        for j in range(i + 1, len(groups)):
            g1 = groups[i]
            g2 = groups[j]
            p_adj = dunn_results.loc[g1, g2]

            if p_adj < alpha:
                decision = "Reject H0"
                interpretation = f"{g1} and {g2} differ significantly"
            else:
                decision = "Fail to reject H0"
                interpretation = f"Insufficient evidence that {g1} and {g2} differ"

            pairwise_decisions.append({
                "Comparison": f"{g1} vs {g2}",
                "Adjusted p-value": p_adj,
                "Decision": decision,
                "Interpretation": interpretation
            })

    decisions_df = pd.DataFrame(pairwise_decisions)
    display(decisions_df)
else:
    print("No pairwise post-hoc decisions because the overall Kruskal-Wallis test was not significant.")

## 5. Complete interpretation

If the Kruskal–Wallis p-value is below 0.05:

> There is statistically significant evidence that the distributions of the three strategies are not all the same.

Then use Dunn's test to determine which pairs differ.

For example:

- A vs B significant → evidence that A and B differ.
- A vs C significant → evidence that A and C differ.
- B vs C not significant → insufficient evidence that B and C differ.

A statistically significant result does **not** automatically mean that one strategy is more profitable. Also examine effect size, direction of the difference, volatility, Sharpe ratio, drawdown, transaction costs, and out-of-sample performance.

## 6. Important note for financial time series

The ordinary Kruskal–Wallis test assumes independent observations. Daily strategy returns can exhibit **autocorrelation, volatility clustering, and other time-series dependence**.

Therefore, for a rigorous strategy comparison, check whether the independence assumption is reasonable and consider an appropriate time-series/resampling methodology when it is not.

Also, Kruskal–Wallis is a rank-based test of distributions. A significant result should not automatically be described as a difference in means.